# Academic calendar and public safety
## A4 · Fall 2024 event-day comparisons

For each home game, move-in day, Halloween, and finals date, compare the recorded collision count with the mean on eligible dates in the same year, month, and weekday. Exclude all named event and holiday flags from the ordinary-day pool; semester membership alone is eligible. Restrict comparisons to dates in the supplied calendar. Dates can serve as controls for more than one event, so the pairs are not independent.

In [ ]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
inputs = root / "data"
calendar = pd.read_csv(inputs / "calendar/daily_calendar.csv", parse_dates=["date"])
calendar = calendar.set_index("date", verify_integrity=True).sort_index()
collisions = pd.read_csv(inputs / "processed/collisions.csv", usecols=["Date", "alcohol_pcf_status"])
collisions["Date"] = pd.to_datetime(collisions["Date"], format="%Y-%m-%d")
print(f"Snapshot records without a local date: {collisions['Date'].isna().sum()}")

In [ ]:
events = ["home_football", "move_in", "halloween", "finals"]
excluded = events + ["holiday", "non_instructional", "rrr", "instruction_begins",
                     "formal_classes_end", "last_instruction_day", "winter_break"]
ordinary = calendar.index[calendar[excluded].eq(0).all(axis=1)]

# Zero here means zero dated records flagged in this snapshot on that day.
flagged = collisions.loc[collisions["alcohol_pcf_status"].eq("explicit_alcohol")]
counts = flagged.groupby("Date").size().reindex(calendar.index, fill_value=0)

records = []
for event in events:
    for day in calendar.index[calendar[event].eq(1)]:
        controls = ordinary[(ordinary.to_period("M") == day.to_period("M"))
                            & (ordinary.weekday == day.weekday())]
        records.append({
            "event": event,
            "event_date": day.date().isoformat(),
            "control_dates": ", ".join(controls.strftime("%Y-%m-%d")),
            "n_controls": len(controls),
            "event_collisions": counts.at[day],
            "mean_control_collisions": counts.loc[controls].mean(),
        })

matches = pd.DataFrame(records)
output = root / "data/processed/a4_collision_matches.csv"
matches.to_csv(output, index=False)
matches

In [ ]:
# Each matched event date receives equal weight, regardless of its pool size.
matched = matches.loc[matches["n_controls"].gt(0)]
summary = matches.groupby("event").agg(
    event_days=("event_date", "size"),
    all_event_collisions=("event_collisions", "sum"),
)
summary.join(matched.groupby("event").agg(
    matched_days=("event_date", "size"),
    mean_event_count=("event_collisions", "mean"),
    mean_control_count=("mean_control_collisions", "mean"),
))

### Interpretation

All seven home games had eligible comparisons. Two flagged collisions occurred on game days. Move-in on August 21 and finals on December 20 had no eligible comparison date; their control means are missing. The sparse counts describe recorded primary factors and support only a descriptive comparison for this semester. Noise and assault comparisons require the shared inputs and their coverage checks.